# Assignment A5 — Pandas exploration (Open Food Facts breakfast cereals)

This notebook loads product-level nutrition data for the **breakfast cereals** category. It uses pandas for exploratory analysis: structure of the table, common categories, subsets, grouped averages, and missing values.

**Dataset Load strategy:** 
1. Paginate the legacy JSON search API until **about 450 rows** (within the planned **400–600** band). 
2. Host order is production (`world.openfoodfacts.org`), then staging (`world.openfoodfacts.net`) if production returns errors—staging uses the same API shape and real product records. 
3. If every host fails, fall back to the bundled `cereals_breakfast.csv` snapshot so the notebook still runs.

In [1]:
import time
import warnings

import pandas as pd
import requests

# Keep notebook output clean from pandas future/deprecation notices.
warnings.filterwarnings("ignore", category=FutureWarning)

# Identify this notebook to the Open Food Facts API.
OFF_USER_AGENT = "HCDE530-assignment/1.0 (https://github.com/openfoodfacts/openfoodfacts-python)"

# Category tag used by the OFF search endpoint.
CATEGORY_TAG = "breakfast-cereals"

# Local CSV snapshot used for fallback/reproducible runs.
SNAPSHOT_CSV = "cereals_breakfast.csv"

# Target sample size (assignment plan: ~400-600 products).
TARGET_FETCH = 450

# Primary + backup OFF hosts; try in order if one fails.
OFF_BASES = [
    "https://world.openfoodfacts.org",
    "https://world.openfoodfacts.net",
]

In [2]:
def flatten_product(p: dict):
    """Map one OFF product JSON object to a flat row for pandas."""
    name = p.get("product_name") or p.get("product_name_en")
    if not name:
        return None
    brands = p.get("brands") or ""
    grade = p.get("nutrition_grade_fr")
    countries = p.get("countries") or ""
    n = p.get("nutriments") or {}
    return {
        "product_name": name,
        "brands": brands.split(",")[0].strip() if brands else "",
        "nutrition_grade_fr": grade,
        "countries": countries.split(",")[0].strip() if countries else "",
        "sugars_100g": n.get("sugars_100g"),
        "fiber_100g": n.get("fiber_100g"),
        "salt_100g": n.get("salt_100g"),
        "saturated-fat_100g": n.get("saturated-fat_100g"),
        "energy-kcal_100g": n.get("energy-kcal_100g"),
    }


def _fetch_search_page(session, base: str, page: int, page_size: int) -> dict:
    fields = "product_name,brands,nutrition_grade_fr,nutriments,countries"
    url = (
        f"{base}/cgi/search.pl"
        f"?action=process&json=true&page_size={page_size}&page={page}"
        f"&tagtype_0=categories&tag_contains_0=contains&tag_0={CATEGORY_TAG}"
        f"&fields={fields}"
    )
    r = session.get(url, timeout=90)
    r.raise_for_status()
    text = r.text.strip()
    if not text.startswith("{"):
        raise ValueError(f"non-JSON response from {base}")
    return r.json()


def fetch_cereals(max_products: int, page_size: int = 100, pause_s: float = 0.4):
    """Paginate OFF legacy JSON search for breakfast cereals; try each host in OFF_BASES per page."""
    rows: list[dict] = []
    page = 1
    session = requests.Session()
    session.headers.update({"User-Agent": OFF_USER_AGENT})
    last_base_used = None

    while len(rows) < max_products:
        payload = None
        errors = []
        for base in OFF_BASES:
            try:
                payload = _fetch_search_page(session, base, page, page_size)
                last_base_used = base
                break
            except Exception as err:
                errors.append(f"{base}: {err!s}")

        if payload is None:
            raise RuntimeError("All OFF hosts failed: " + " | ".join(errors))

        products = payload.get("products") or []
        if not products:
            break
        for p in products:
            flat = flatten_product(p)
            if flat:
                rows.append(flat)
            if len(rows) >= max_products:
                break
        page += 1
        time.sleep(pause_s)

    return pd.DataFrame(rows), last_base_used

In [3]:
# Load MP1 dataset: prefer live Open Food Facts pull; use CSV snapshot if every host fails.
try:
    df, host_used = fetch_cereals(max_products=TARGET_FETCH)
    if len(df) == 0:
        raise RuntimeError("API returned no products")
    df.to_csv(SNAPSHOT_CSV, index=False)
    print(
        f"Loaded {len(df)} products via {host_used} and refreshed {SNAPSHOT_CSV} "
        f"(target band ~400–600; fetch cap={TARGET_FETCH})."
    )
except Exception as exc:
    df = pd.read_csv(SNAPSHOT_CSV)
    print(f"API fetch failed ({type(exc).__name__}: {exc}). Using bundled snapshot {SNAPSHOT_CSV} ({len(df)} rows).")

df.shape

Loaded 450 products via https://world.openfoodfacts.net and refreshed cereals_breakfast.csv (target band ~400–600; fetch cap=450).


(450, 9)

# **Operation 1:** 
`head()` and `info()`

Inspect column names, dtypes, and a sample of rows.

In [4]:
# I'm asking: what columns exist, what types does pandas infer, and what do a few cereal rows look like?
# The answer tells me whether nutrition fields parsed as numbers, whether brand/country text looks usable, and rough scale of the dataset before deeper analysis.
df.head(8)

,product_name,brands,nutrition_grade_fr,countries,sugars_100g,fiber_100g,salt_100g,saturated-fat_100g,energy-kcal_100g
0,Weetabix,Weetabix,a,France,4.210526,10.000000,0.256579,0.526316,357.894737
1,cruesly mélange de noix,Quaker,b,Belgium,12.000000,10.000000,0.000000,2.000000,462.000000
2,Weetabix,Weetabix,a,France,4.000000,9.500000,0.243750,0.500000,340.000000
3,Flocons avoine,LIDL,a,Belgien,0.700000,10.000000,0.030000,1.300000,372.000000
4,Flocons d'avoine,Bjorg,a,Belgium,1.700000,11.000000,0.020000,1.300000,362.000000
5,Croustillant Chocolat,Bjorg,c,France,14.000000,10.000000,0.266667,3.333333,431.666667
6,Weetabix produit à base de blé complet 100%,Weetabix,a,Belgique,4.266667,10.133333,0.260000,0.533333,362.666667
7,Wheat Bisks,Harvest Morn,a,United Kingdom,4.473684,10.000000,0.256579,0.526316,353.000000


In [5]:
# I'm asking: are there missing values already visible per column, and which columns are numeric vs object?
# The answer means I know memory footprint, non-null counts, and whether I need to coerce dtypes before filtering or aggregation.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 450 entries, 0 to 449
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   product_name        450 non-null    str    
 1   brands              450 non-null    str    
 2   nutrition_grade_fr  450 non-null    str    
 3   countries           450 non-null    str    
 4   sugars_100g         441 non-null    float64
 5   fiber_100g          436 non-null    float64
 6   salt_100g           438 non-null    float64
 7   saturated-fat_100g  439 non-null    float64
 8   energy-kcal_100g    441 non-null    float64
dtypes: float64(5), str(4)
memory usage: 31.8 KB


## `value_counts()`

In [6]:
# I'm asking: which Nutri-Score grades appear most often in this cereal slice?
# The answer tells me how imbalanced the label distribution is (e.g. many C/D vs few A), which affects any later comparison across grades.
df["nutrition_grade_fr"].value_counts(dropna=False)

nutrition_grade_fr
c          159
a          146
d           63
b           58
e           17
unknown      7
Name: count, dtype: int64

# **Operation:** 
Filter to a subset

In [7]:
# I'm asking: among cereals with measured sugar, which ones report more than 20 g sugar per 100 g?
# The answer highlights higher-sugar products in the subset so I can spot extremes rather than averaging everything together.
high_sugar = df[df["sugars_100g"] > 20]
high_sugar[["product_name", "brands", "sugars_100g", "nutrition_grade_fr"]].head(12)

,product_name,brands,sugars_100g,nutrition_grade_fr
24,Céréales Chocapic,Nestlé,22.4,c
26,Country Crisp - Chocolat noir 70% cacao,Jordans,22.0,d
63,Alpenmüsli The Original Swiss Style,Weetabix,21.0,c
64,Crunchy Oat Granola Raisin & Almond,Jordans,21.0,c
66,Miel pops,KELLOG'S,22.0,c
76,Premium Fruit & Nut Muesli,Lidl,24.6,c
83,Nesquik Cereal,Nesquik,22.4,unknown
87,Céréales Miel Pops,Kellogg's,22.0,c
89,Granola chocolate / pieces of chocolate,Sante,22.0,d
91,Tresor - Choco Nut,Kellogg's,26.0,d


# **Operation:** 
`groupby` + summary (`mean`)

In [8]:
# I'm asking: for each brand in the sample, what is the average reported sugar (per 100 g)?
# The answer ranks brands by typical sugar level in this category (within this dataset), useful for comparing positioning across manufacturers.
(
    df.groupby("brands")["sugars_100g"]
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

brands
Lucien Georgelin             32.000000
Break & Boost                30.000000
OREO                         27.000000
Nature Valley                26.190476
Terres et Céréales Bio       26.000000
T&C Terres & Céréales BIO    24.000000
Leclerc                      24.000000
Kellogg’s                    22.500000
Nesquik                      22.400000
Caterers pride               22.400000
KELLOG'S                     22.000000
Maison Pâtissière            22.000000
                             21.825000
Joseph Favrichon             21.111111
Kellogg's                    20.415062
Name: sugars_100g, dtype: float64

# **Operation:**
Missing data: `isnull().sum()`

In [9]:
# I'm asking: which nutrition columns are missing most often, i.e. how complete is macronutrient reporting?
# The answer shows where dashboards or apps would lack trustworthy fields (more NaNs ⇒ more placeholders or gaps for users).
nutrition_cols = [
    "sugars_100g",
    "fiber_100g",
    "salt_100g",
    "saturated-fat_100g",
    "energy-kcal_100g",
]
df[nutrition_cols].isnull().sum().sort_values(ascending=False)

fiber_100g            14
salt_100g             12
saturated-fat_100g    11
sugars_100g            9
energy-kcal_100g       9
dtype: int64

# **Analytical Question 1:**
Which brands have highest average sugar_100g, and does Nutri-Score track sugar reliably?

In [1]:
# Block 1 — Q1:
# "Which brands have highest average sugar_100g, and does Nutri-Score track sugar reliably?"

import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("../week-6/cereals_breakfast.csv")

# Basic cleanup
df["brands"] = df["brands"].astype(str).str.strip()
df["nutrition_grade_fr"] = df["nutrition_grade_fr"].astype(str).str.strip().str.lower()
df["sugars_100g"] = pd.to_numeric(df["sugars_100g"], errors="coerce")

# Keep rows with valid brand + sugar
q1 = df.dropna(subset=["brands", "sugars_100g"]).copy()

# Top brands by average sugar (add count so tiny samples are visible)
brand_sugar = (
    q1.groupby("brands", as_index=False)
      .agg(
          n_products=("product_name", "count"),
          avg_sugars_100g=("sugars_100g", "mean")
      )
      .sort_values("avg_sugars_100g", ascending=False)
)

print("Top 15 brands by average sugar (g/100g):")
print(brand_sugar.head(15).to_string(index=False))

# Nutri-Score reliability vs sugar
grade_order = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
q1_grade = q1[q1["nutrition_grade_fr"].isin(grade_order)].copy()
q1_grade["grade_num"] = q1_grade["nutrition_grade_fr"].map(grade_order)

# Average sugar by grade
grade_sugar = (
    q1_grade.groupby("nutrition_grade_fr", as_index=False)["sugars_100g"]
            .mean()
            .rename(columns={"sugars_100g": "avg_sugars_100g"})
)
grade_sugar["grade_num"] = grade_sugar["nutrition_grade_fr"].map(grade_order)
grade_sugar = grade_sugar.sort_values("grade_num")

print("\nAverage sugar by Nutri-Score grade:")
print(grade_sugar[["nutrition_grade_fr", "avg_sugars_100g"]].to_string(index=False))

# Correlation: higher grade_num (worse score) should ideally mean higher sugar
corr = q1_grade["grade_num"].corr(q1_grade["sugars_100g"], method="spearman")
print(f"\nSpearman correlation (grade_num vs sugars_100g): {corr:.3f}")

# Optional: quick overlap diagnostics using quartiles
q1_grade["sugar_quartile"] = pd.qcut(q1_grade["sugars_100g"], q=4, labels=["Q1_low", "Q2", "Q3", "Q4_high"])
cross = pd.crosstab(q1_grade["nutrition_grade_fr"], q1_grade["sugar_quartile"], normalize="index")
print("\nGrade vs sugar quartiles (row-normalized):")
print(cross.round(3).to_string())

Top 15 brands by average sugar (g/100g):
                   brands  n_products  avg_sugars_100g
         Lucien Georgelin           1        32.000000
            Break & Boost           1        30.000000
                     OREO           1        27.000000
            Nature Valley           1        26.190476
   Terres et Céréales Bio           1        26.000000
                  Leclerc           2        24.000000
T&C Terres & Céréales BIO           1        24.000000
                Kellogg’s           2        22.500000
                  Nesquik           1        22.400000
           Caterers pride           1        22.400000
                 KELLOG'S           1        22.000000
        Maison Pâtissière           1        22.000000
         Joseph Favrichon           2        21.111111
                Kellogg's          45        20.415062
       Terres et Céréales           1        20.000000

Average sugar by Nutri-Score grade:
nutrition_grade_fr  avg_sugars_100g
      

# **Analytical Question 2:**
What share of products with health-oriented keywords in name have Nutri-Score C or worse?

In [2]:
# Block 2 — Q2:
# "What share of products with health-oriented keywords in name have Nutri-Score C or worse?"

import pandas as pd

df = pd.read_csv("../week-6/cereals_breakfast.csv")

# Cleanup
df["product_name"] = df["product_name"].astype(str).str.strip()
df["nutrition_grade_fr"] = df["nutrition_grade_fr"].astype(str).str.strip().str.lower()

# Keyword filter (case-insensitive)
# Includes "whole grain", "natural", "fiber"/"fibre"
keyword_pattern = r"\b(whole\s*grain|natural|fiber|fibre)\b"
health_named = df[df["product_name"].str.contains(keyword_pattern, case=False, regex=True, na=False)].copy()

# Define C or worse
c_or_worse = {"c", "d", "e"}
known_grades = {"a", "b", "c", "d", "e"}

health_named["is_known_grade"] = health_named["nutrition_grade_fr"].isin(known_grades)
health_known = health_named[health_named["is_known_grade"]].copy()
health_known["is_c_or_worse"] = health_known["nutrition_grade_fr"].isin(c_or_worse)

n_total_keywords = len(health_named)
n_known = len(health_known)
n_c_or_worse = int(health_known["is_c_or_worse"].sum())
share_c_or_worse = (n_c_or_worse / n_known) if n_known else float("nan")

print(f"Products with keywords in name: {n_total_keywords}")
print(f"With known Nutri-Score (A-E): {n_known}")
print(f"C or worse among known grades: {n_c_or_worse}")
print(f"Share C or worse: {share_c_or_worse:.2%}")

# Optional: distribution among keyword-matched products
dist = health_known["nutrition_grade_fr"].value_counts(normalize=True).sort_index()
print("\nNutri-Score distribution among keyword-matched products:")
print((dist * 100).round(2).astype(str) + "%")

Products with keywords in name: 9
With known Nutri-Score (A-E): 9
C or worse among known grades: 5
Share C or worse: 55.56%

Nutri-Score distribution among keyword-matched products:
nutrition_grade_fr
a    44.44%
c    44.44%
d    11.11%
Name: proportion, dtype: str


/var/folders/5g/1v2k1rz567qdgmk1lyf9k73m0000gn/T/ipykernel_62627/3534477397.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  health_named = df[df["product_name"].str.contains(keyword_pattern, case=False, regex=True, na=False)].copy()


# **Analytical Question 3:**
Which nutritional fields are most frequently missing, and does completeness vary by country?

In [3]:
# Block 3 — Q3:
# "Which nutritional fields are most frequently missing, and does completeness vary by country?"

import pandas as pd

df = pd.read_csv("../week-6/cereals_breakfast.csv")

# Nutritional fields present in this dataset
nutrition_cols = ["sugars_100g", "fiber_100g", "salt_100g", "saturated-fat_100g", "energy-kcal_100g"]

# Ensure numeric parsing (non-numeric -> NaN, counted as missing)
for col in nutrition_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Overall missingness
missing_overall = (
    df[nutrition_cols]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "field"})
)
print("Overall missingness by field (% of products):")
print(missing_overall.to_string(index=False))

# Country-level completeness
df["countries"] = df["countries"].astype(str).str.strip()

# Keep countries with enough products for stable comparisons (adjust threshold if needed)
min_n = 5
country_counts = df["countries"].value_counts()
valid_countries = country_counts[country_counts >= min_n].index

by_country = (
    df[df["countries"].isin(valid_countries)]
    .groupby("countries")[nutrition_cols]
    .apply(lambda g: g.notna().mean() * 100)
    .round(2)
)

# Add average completeness score across nutritional fields
by_country["avg_completeness_pct"] = by_country.mean(axis=1)
by_country = by_country.sort_values("avg_completeness_pct", ascending=False)

print(f"\nCountry-level completeness (%), countries with n >= {min_n}:")
print(by_country.to_string())

# Optional: where missingness is concentrated (field x country)
missing_country_field = (
    df[df["countries"].isin(valid_countries)]
    .groupby("countries")[nutrition_cols]
    .apply(lambda g: g.isna().mean() * 100)
    .round(2)
)
print("\nMissingness (%) by country and field:")
print(missing_country_field.to_string())

Overall missingness by field (% of products):
             field  missing_pct
        fiber_100g     3.111111
         salt_100g     2.666667
saturated-fat_100g     2.444444
       sugars_100g     2.000000
  energy-kcal_100g     2.000000

Country-level completeness (%), countries with n >= 5:
                   sugars_100g  fiber_100g  salt_100g  saturated-fat_100g  energy-kcal_100g  avg_completeness_pct
countries                                                                                                        
Austria                 100.00      100.00     100.00              100.00            100.00               100.000
Belgique                100.00      100.00     100.00              100.00            100.00               100.000
Bélgica                 100.00      100.00     100.00              100.00            100.00               100.000
España                  100.00      100.00     100.00              100.00            100.00               100.000
Portugal              